# Multiburst InSAR Capability

Last year, ASF launched the ability to create on-demand InSAR pairs from single Sentinel-1 bursts. We've heard a lot of feedback from our users that they like this capability, and we've continued to improve our burst offerings throughout the year. Several important updates include:

1. Full back-population of the burst archive. You can now download burst data for any Sentinel-1 scene in ASF's archive!
2. Creation of the [`burst2safe`](https://github.com/forrestfwilliams/burst2safe) Python package, which allows users to compose custom SAFE files from sets of Sentinel-1 bursts.
3. A Multiburst InSAR command line tool, which this notebook discusses in greater detail.

Later this year, we plan to extend our burst InSAR HyP3 job to be able to create InSAR products from multiple bursts. This will be a huge improvement in the utility of the HyP3 job because most deformation patterns span somewhere between 1 and 20 bursts.

As a first step towards offering this capability via HyP3, we've added the ability to perform multiburst InSAR to our [`HyP3-ISCE2`](https://github.com/asfhyp3/hyp3-isce2) package, which you can run for yourself today. This notebook demos the current functionality.

For this notebook, we will be comparing multiburst and standard SLC InSAR for the [2019 Ridgecrest Earthquake](https://earthquake.usgs.gov/storymap/index-ridgecrest.html) using the ascending pass of Sentinel-1.

## Setup

First, let's set up some directories for running the analyses

In [1]:
from pathlib import Path


base = Path.cwd()
slc_upper = base / 'slc_upper'
slc_lower = base / 'slc_lower'
burst = base / 'burst'

for directory in [slc_upper, slc_lower, burst]:
    directory.mkdir(parents=True, exist_ok=True)

## SLC

Let's take a look at how to create ascending pass Sentinel-1 InSAR data for the Ridgecrest Earthquake.

The Ridgecrest Earthquake deformation is located at the boundary between two Sentinel-1 SLC frames, so we'll have to produce two InSAR pairs. This leads to longer runtimes and producing InSAR data for areas we're not interested in. 

## Upper SLC Processing

<a href="https://search.asf.alaska.edu/#/?maxResults=250&zoom=7.504&center=-114.798,33.203&polygon=POLYGON((-117.8581%2035.2613,-117.0303%2035.2613,-117.0303%2035.9371,-117.8581%2035.9371,-117.8581%2035.2613))&searchType=Geographic%20Search&resultsLoaded=true&granule=S1A_IW_SLC__1SDV_20190716T015049_20190716T015116_028136_032D86_6DF2-SLC&productTypes=SLC&flightDirs=Ascending&start=2019-07-03T05:00:00Z&end=2019-07-25T04:59:59Z&path=64-64&frame=114-114" > View the input Sentinel-1 scenes in Vertex  </a>

The command below takes about 60 minutes to run when the call to `hyp3_isce2` is uncommented.

In [2]:
%%bash -s "$slc_upper"
cd $1
# python -m hyp3_isce2 ++process insar_tops \
#     --reference S1A_IW_SLC__1SDV_20190704T015049_20190704T015116_027961_03283A_191E \
#     --secondary S1B_IW_SLC__1SDV_20190710T014959_20190710T015026_017065_0201B8_069B
du -sh $1

 41G	/Users/ffwilliams2/Repositories/hyp3/tutorials/InSAR/multiburst/slc_upper


![Top SLC InSAR Pair](./assets/top_slc.png "Top SLC InSAR Pair")

## Lower SLC Processing

<a href="https://search.asf.alaska.edu/#/?maxResults=250&zoom=7.504&center=-114.798,33.203&polygon=POLYGON((-117.8581%2035.2613,-117.0303%2035.2613,-117.0303%2035.9371,-117.8581%2035.9371,-117.8581%2035.2613))&searchType=Geographic%20Search&resultsLoaded=true&granule=S1A_IW_SLC__1SDV_20190704T015023_20190704T015051_027961_03283A_D6EA-SLC&productTypes=SLC&flightDirs=Ascending&start=2019-07-03T05:00:00Z&end=2019-07-25T04:59:59Z&path=64-64&frame=109-109" > View the input Sentinel-1 scenes in Vertex. </a>

The command below takes about 30 minutes to run when the call to `hyp3_isce2` is uncommented.

In [3]:
%%bash -s "$slc_lower"
cd $1
# python -m hyp3_isce2 ++process insar_tops \
#     --reference S1A_IW_SLC__1SDV_20190704T015023_20190704T015051_027961_03283A_D6EA \
#     --secondary S1B_IW_SLC__1SDV_20190710T014959_20190710T015026_017065_0201B8_069B
du -sh $1

 21G	/Users/ffwilliams2/Repositories/hyp3/tutorials/InSAR/multiburst/slc_lower


![Bottom SLC InSAR Pair](./assets/bottom_slc.png "Bottom SLC InSAR Pair")

From here, you'd have to merge the two InSAR pairs, which is very challenging due to differences in the reference point and unwrapping decisions made for each pair. The lack of the ability to merge InSAR pairs after processing is a drawback of current SLC-based HyP3 InSAR job.

## Multiburst Processing

<a href="https://search.asf.alaska.edu/#/?maxResults=250&zoom=8.511&center=-118.003,34.754&polygon=POLYGON((-117.5715%2035.3207,-117.1319%2035.3207,-117.1319%2035.9412,-117.5715%2035.9412,-117.5715%2035.3207))&searchType=Geographic%20Search&resultsLoaded=true&granule=S1_135528_IW2_20190716T015049_VV_6DF2-BURST&path=64-64&frame=109-109&flightDirs=Ascending&start=2019-07-15T05:00:00Z&end=2019-07-18T04:59:59Z&dataset=SENTINEL-1%20BURSTS&polarizations=VV" > View input Sentinel-1 scenes in Vertex. </a>

With multiburst InSAR, we can define our own custom burst set that FULLY AND ONLY covers the area of earthquake-related deformation.

The command below takes about 20 minutes to run when the call to `hyp3_isce2` is uncommented.

In [4]:
%%bash -s "$burst"
cd $1
# python -m hyp3_isce2 ++process insar_tops_burst \
#     --reference \
#         S1_135529_IW3_20190704T015052_VV_191E-BURST \
#         S1_135529_IW2_20190704T015051_VV_191E-BURST \
#         S1_135528_IW3_20190704T015050_VV_191E-BURST \
#         S1_135528_IW2_20190704T015049_VV_191E-BURST \
#         S1_135527_IW3_20190704T015047_VV_D6EA-BURST \
#         S1_135527_IW2_20190704T015046_VV_D6EA-BURST \
#         S1_135526_IW3_20190704T015044_VV_D6EA-BURST \
#         S1_135526_IW2_20190704T015043_VV_D6EA-BURST \
#         S1_135525_IW3_20190704T015041_VV_D6EA-BURST \
#         S1_135525_IW2_20190704T015040_VV_D6EA-BURST \
#     --secondary \
#         S1_135529_IW3_20190710T015011_VV_069B-BURST \
#         S1_135529_IW2_20190710T015010_VV_069B-BURST \
#         S1_135528_IW3_20190710T015008_VV_069B-BURST \
#         S1_135528_IW2_20190710T015007_VV_069B-BURST \
#         S1_135527_IW3_20190710T015006_VV_069B-BURST \
#         S1_135527_IW2_20190710T015005_VV_069B-BURST \
#         S1_135526_IW3_20190710T015003_VV_069B-BURST \
#         S1_135526_IW2_20190710T015002_VV_069B-BURST \
#         S1_135525_IW3_20190710T015000_VV_069B-BURST \
#         S1_135525_IW2_20190710T014959_VV_069B-BURST
du -sh $1

 27G	/Users/ffwilliams2/Repositories/hyp3/tutorials/InSAR/multiburst/burst


![Multiburst InSAR Pair](./assets/multiburst.png "Multiburst InSAR Pair")

Notice that the multiburst processing provides us with the data we actually want, AND does it in half the time, AND produces half of the data volume!